# TR Smart-Home LLM Benchmark
**CSE 440 – Final Project**

### Before running:
1. `Runtime → Change runtime type → T4 GPU`
2. Get a HuggingFace token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Accept model licenses:
   - [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
   - [google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it)

---
## 1 — Setup

In [ ]:
# Mount Google Drive to save results permanently
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo and move into it
!git clone https://github.com/enesdemir0/tr-smart-home-llm-benchmark.git
%cd tr-smart-home-llm-benchmark

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# HuggingFace login (required for LLaMA and Gemma gated models)
from huggingface_hub import login
login(token="hf_YOUR_TOKEN_HERE")  # <- paste your token here

In [ ]:
# Verify GPU is available
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

---
## 2 — Fine-Tuning
> **Run once per model.** Save results to Drive immediately after — if the session disconnects the adapter is lost.

In [ ]:
# Fine-tune LLaMA 3.2-3B (~35 Colab units)
!python run.py --model llama --mode finetune --action train --epochs 3

In [ ]:
import shutil, os
os.makedirs("/content/drive/MyDrive/smart_home_results", exist_ok=True)
shutil.copytree("results/llama_finetuned", "/content/drive/MyDrive/smart_home_results/llama_finetuned", dirs_exist_ok=True)
print("LLaMA adapter saved to Drive.")

In [ ]:
# Fine-tune Gemma 2-2B (~30 Colab units)
!python run.py --model gemma --mode finetune --action train --epochs 3

In [ ]:
import shutil, os
os.makedirs("/content/drive/MyDrive/smart_home_results", exist_ok=True)
shutil.copytree("results/gemma_finetuned", "/content/drive/MyDrive/smart_home_results/gemma_finetuned", dirs_exist_ok=True)
print("Gemma adapter saved to Drive.")

---
## 3 — Evaluation

Each cell runs one model × mode × prompt combination.  
Results are automatically appended to `results/summary.csv`.

In [ ]:
# Restore adapters from Drive (run this if session restarted after fine-tuning)
import shutil, os

drive_base = "/content/drive/MyDrive/smart_home_results"
os.makedirs("results", exist_ok=True)

for model_key in ["llama", "gemma"]:
    src = f"{drive_base}/{model_key}_finetuned"
    dst = f"results/{model_key}_finetuned"
    if os.path.exists(f"{src}/adapter_config.json"):
        if not os.path.exists(f"{dst}/adapter_config.json"):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"{model_key}: adapter restored from Drive -> {dst}")
        else:
            print(f"{model_key}: adapter already in results/ (skipped)")
    else:
        print(f"{model_key}: not on Drive yet (fine-tuning needed)")


### 3.1 — LLaMA Base (no fine-tuning)

In [ ]:
!python run.py --model llama --mode base --prompt zero_shot

In [ ]:
!python run.py --model llama --mode base --prompt few_shot

In [ ]:
!python run.py --model llama --mode base --prompt cot

### 3.2 — LLaMA Fine-Tuned

In [ ]:
!python run.py --model llama --mode finetune --prompt zero_shot

In [ ]:
!python run.py --model llama --mode finetune --prompt few_shot

In [ ]:
!python run.py --model llama --mode finetune --prompt cot

### 3.3 — Gemma Base (no fine-tuning)

In [ ]:
!python run.py --model gemma --mode base --prompt zero_shot

In [ ]:
!python run.py --model gemma --mode base --prompt few_shot

In [ ]:
!python run.py --model gemma --mode base --prompt cot

### 3.4 — Gemma Fine-Tuned

In [ ]:
!python run.py --model gemma --mode finetune --prompt zero_shot

In [ ]:
!python run.py --model gemma --mode finetune --prompt few_shot

In [ ]:
!python run.py --model gemma --mode finetune --prompt cot

### 3.5 — General LLM (Mistral-7B, no fine-tuning)

In [ ]:
!python run.py --model general --mode base --prompt zero_shot

In [ ]:
!python run.py --model general --mode base --prompt few_shot

In [ ]:
!python run.py --model general --mode base --prompt cot

---
## 4 — Results

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary.csv")

# Round for clean display
pct_cols = ["valid_json_rate", "label_validity_rate", "accuracy", "precision", "recall", "f1"]
for col in pct_cols:
    if col in df.columns:
        df[col] = (df[col] * 100).round(1).astype(str) + "%"

df

In [ ]:
# Plot accuracy comparison across all configurations
import matplotlib.pyplot as plt
import pandas as pd

df_raw = pd.read_csv("results/summary.csv")
df_raw["label"] = df_raw["model"] + "\n" + df_raw["mode"] + "\n" + df_raw["prompt"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
axes[0].bar(df_raw["label"], df_raw["accuracy"] * 100, color="steelblue")
axes[0].set_title("Accuracy (%)", fontsize=13)
axes[0].set_ylim(0, 100)
axes[0].tick_params(axis="x", rotation=90, labelsize=7)
axes[0].set_ylabel("%")

# F1
axes[1].bar(df_raw["label"], df_raw["f1"] * 100, color="darkorange")
axes[1].set_title("F1 Score (%)", fontsize=13)
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].set_ylabel("%")

plt.tight_layout()
plt.savefig("results/accuracy_f1_comparison.png", dpi=150)
plt.show()
print("Plot saved to results/accuracy_f1_comparison.png")

In [ ]:
# Token usage and latency comparison
import matplotlib.pyplot as plt
import pandas as pd

df_raw = pd.read_csv("results/summary.csv")
df_raw["label"] = df_raw["model"] + "\n" + df_raw["mode"] + "\n" + df_raw["prompt"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total tokens
axes[0].bar(df_raw["label"], df_raw["avg_total_tokens"], color="mediumpurple")
axes[0].set_title("Avg Total Tokens", fontsize=13)
axes[0].tick_params(axis="x", rotation=90, labelsize=7)
axes[0].set_ylabel("tokens")

# Latency
axes[1].bar(df_raw["label"], df_raw["avg_latency_s"], color="tomato")
axes[1].set_title("Avg Latency (s)", fontsize=13)
axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].set_ylabel("seconds")

plt.tight_layout()
plt.savefig("results/token_latency_comparison.png", dpi=150)
plt.show()
print("Plot saved to results/token_latency_comparison.png")

---
## 5 — Save Everything to Drive

In [ ]:
import shutil, os

drive_path = "/content/drive/MyDrive/smart_home_results"
os.makedirs(drive_path, exist_ok=True)

# Copy entire results folder
shutil.copytree("results", drive_path, dirs_exist_ok=True)
print(f"All results saved to {drive_path}")